# Measuring the Sun's Angular Diameter Using a Smart Telescope

This notebook calculates the angular diameter of the Sun using data collected from a "drift transit" experiment using a ZWO Seestar S50 telescope. Because the telescope remains stationary while the Earth rotates, the Sun drifts across the camera sensor. 

We will use trigonometry to correct for the diagonal angle of the Sun's path across the sensor and apply a correction for the Sun's current celestial declination.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURABLE EXPERIMENTAL DATA ---
# Replace these values with your actual measurements from the Seestar video

delta_x = 800       # Horizontal pixel displacement of the drift path
delta_y = 450       # Vertical pixel displacement of the drift path
t_measured = 152.5  # Observed transit time in seconds (leading edge to trailing edge)
declination_deg = 20.8  # Sun's declination in degrees for late May
# --------------------------------------

print("Experimental data loaded successfully.")

## Step 1: Visualizing the Drift Angle and Vector Geometry

Before calculating, let's visualize how the Sun moves across the rectangular sensor array. The true path of the Sun forms the hypotenuse of a right triangle created by the horizontal ($\Delta X$) and vertical ($\Delta Y$) pixel movements.

In [ ]:
# Calculate the drift angle (alpha) in radians and degrees
alpha_rad = np.arctan(delta_y / delta_x)
alpha_deg = np.degrees(alpha_rad)

# Plotting the geometric vector
plt.figure(figsize=(8, 5))
plt.plot([0, delta_x], [0, delta_y], 'ro-', label="Sun's Drift Path (Hypotenuse)", linewidth=2)
plt.axhline(0, color='black', linestyle='--', label="Sensor Horizontal Axis (X)")
plt.axvline(delta_x, color='gray', linestyle=':', label="Sensor Vertical Axis (Y)")

# Annotate the plot
plt.text(delta_x / 2, delta_y / 2 + 30, "True Path", fontsize=11, color='red')
plt.text(delta_x * 0.2, 15, f"Angle α = {alpha_deg:.2f}°", fontsize=12, fontweight='bold')
plt.title("Geometry of the Sun's Diagonal Drift Across the Seestar Sensor")
plt.xlabel("Horizontal Pixels (ΔX)")
plt.ylabel("Vertical Pixels (ΔY)")
plt.xlim(-50, delta_x + 100)
plt.ylim(-50, delta_y + 100)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc="upper left")
plt.show()

print(f"Calculated Drift Angle (α): {alpha_deg:.2f} degrees")

## Step 2: Correcting the Transit Time

Because the Sun travels at a diagonal angle $\alpha$, its apparent transit time across a fixed vertical reference line is stretched out. We calculate the true perpendicular transit time ($T_{\text{true}}$) across its own diameter using the cosine of our drift angle:

$$T_{\text{true}} = T_{\text{measured}} \times \cos(\alpha)$$

In [ ]:
# Calculate the corrected transit time
t_true = t_measured * np.cos(alpha_rad)

print(f"Measured Transit Time: {t_measured:.2f} seconds")
print(f"Corrected Perpendicular Transit Time (T_true): {t_true:.2f} seconds")

## Step 3: Calculating Angular Diameter with Declination Correction

The Earth rotates $360^\circ$ relative to the Sun in approximately 24 hours (86,400 seconds), yielding an uncorrected angular speed of:

$$\omega = \frac{360^\circ}{86400 \text{ s}} \approx 0.004167^\circ/\text{s}$$

Because we are observing from a mid-latitude location, the Sun's path shrinks along a smaller circle of latitude as its declination increases. We apply the cosine of the Sun's declination ($\delta$) to find the true angular diameter ($\theta$):

$$\theta = (T_{\text{true}} \times 0.004167^\circ/\text{s}) \times \cos(\delta)$$

In [ ]:
# Constants
earth_rotation_speed = 360.0 / 86400.0

# Convert declination to radians
declination_rad = np.radians(declination_deg)

# Calculate Angular Diameter in degrees
angular_diameter_deg = (t_true * earth_rotation_speed) * np.cos(declination_rad)

# Convert to arcminutes (1 degree = 60 arcminutes)
angular_diameter_arcmin = angular_diameter_deg * 60.0

# Print final results
print("============= FINAL RESULTS =============")
print(f"Calculated Angular Diameter: {angular_diameter_deg:.4f}°")
print(f"Calculated Angular Diameter: {angular_diameter_arcmin:.2f} arcminutes")
print("=========================================")
print("Note: The accepted true value fluctuates around ~32 arcminutes (~0.533°).")